# 🔴 VOIS RAG Hands-On Lab
### Build a RAG Q&A Bot — Step by Step

**What you will build:** A Q&A system over a fictional VOIS Employee Handbook.

| Cell | What you do |
|------|-------------|
| 1 | Install packages |
| 2 | Load the knowledge base |
| 3 | Chunk documents |
| 4 | Embed + store in vector DB |
| 5 | Ask a question (retrieval only) |
| 6 | Generate an answer with GPT-4o |
| 7 | 🎯 Try This: change chunk size |
| 8 | Hybrid search |
| 9 | Reranking |
| 10 | 🎯 Compare all three approaches |
| 11 | Build eval dataset |
| 12 | Run RAGAS evaluation |
| 13 | How to move to Azure |

> **No API key needed for cells 1–10 and the comparison.**
> Cells 6, 11, 12 work best with an OpenAI key but gracefully skip without one.

---
## ⚙️ Cell 1 — Install Packages
Run once. Takes about 2 minutes on Colab.

In [ ]:
%%capture
!pip install langchain langchain-community langchain-openai \
             chromadb sentence-transformers \
             rank_bm25 openai ragas datasets
print('✅ Done')

---
## 📄 Cell 2 — Load the Knowledge Base
A fictional **VOIS Employee Handbook** — 10 policy documents built into this notebook.
No file upload needed.

In [ ]:
from langchain.schema import Document

raw_docs = [
    Document(page_content="Annual Leave Policy: VOIS employees are entitled to 21 working days of annual leave per calendar year. Leave must be approved by the direct line manager at least 2 weeks in advance. Unused leave of up to 5 days may be carried over to the following year. Employees who join mid-year receive leave on a pro-rata basis. Requests should be submitted through the VOIS HR portal.", metadata={"section": "Leave Policy"}),
    Document(page_content="Sick Leave Policy: VOIS provides up to 14 days of paid sick leave per year. A medical certificate is required for absences exceeding 3 consecutive days. Sick leave does not accumulate and cannot be converted to cash. In cases of prolonged illness more than 30 days, employees may apply for extended medical leave subject to HR and Medical Committee approval.", metadata={"section": "Leave Policy"}),
    Document(page_content="Remote Work Policy: VOIS employees in eligible roles may work remotely up to 3 days per week. Remote work days must be agreed with the line manager at the start of each month. Employees are expected to be available during core hours 10:00 to 15:00 Cairo time regardless of location. New employees in their first 90 days are required to work on-site full-time.", metadata={"section": "Work Arrangements"}),
    Document(page_content="Performance Review Cycle: VOIS follows a bi-annual performance review cycle. Mid-year reviews take place in June and year-end reviews in December. Performance is rated on a 5-point scale: Exceptional, Exceeds Expectations, Meets Expectations, Needs Improvement, and Unsatisfactory. Employees rated Needs Improvement for two consecutive cycles may be placed on a Performance Improvement Plan.", metadata={"section": "Performance"}),
    Document(page_content="Training and Development: VOIS employees are entitled to a personal training budget of EGP 15,000 per year. This budget covers external courses, certifications, or conferences approved by HR. VOIS also provides access to LinkedIn Learning, Coursera for Business, and an internal LMS. Employees seeking certifications such as AWS, Azure, or PMP can apply for full reimbursement via the Certification Sponsorship Program, subject to a 1-year stay-back clause.", metadata={"section": "Development"}),
    Document(page_content="IT Equipment Policy: All employees receive a laptop upon joining. The standard is a Dell Latitude or MacBook Pro depending on role. Employees in AI and Data roles may request a GPU-enabled workstation through the IT portal. Equipment must be returned in good condition upon resignation or termination. Loss or damage due to negligence may result in a financial deduction from the final settlement.", metadata={"section": "Equipment"}),
    Document(page_content="Data Security Policy: All VOIS employees must complete the annual Cybersecurity Awareness training. Customer data must never be stored on personal devices or shared via personal email. Access to production systems requires MFA. Any suspected security incident must be reported to the Security Operations Center within 1 hour. Violation may result in disciplinary action up to and including termination.", metadata={"section": "Security"}),
    Document(page_content="Employee Benefits: VOIS provides comprehensive medical insurance for employees and their immediate family including spouse and up to 3 children. Coverage includes inpatient, outpatient, dental, and optical treatment. Employees also receive life insurance equal to 3 years of annual gross salary. A subsidised meal allowance of EGP 2,500 per month is provided for on-site days. A shuttle bus service covers major Cairo and Giza districts.", metadata={"section": "Benefits"}),
    Document(page_content="Promotion and Career Progression: Career progression at VOIS follows a defined career ladder from Junior L1 to Principal L5. Promotions are reviewed annually in December. Eligibility requires a minimum of 12 months in the current role and a rating of Exceeds Expectations or above. Promotion decisions are made by the direct manager and approved by the department head.", metadata={"section": "Career"}),
    Document(page_content="Code of Conduct: VOIS employees are expected to act with integrity, respect, and professionalism. Harassment, discrimination, or bullying will not be tolerated. Employees must declare any conflict of interest. Accepting gifts from vendors exceeding EGP 500 must be approved by the Ethics Committee. Whistleblower protections are in place for employees who report misconduct in good faith.", metadata={"section": "Conduct"}),
]

print(f'✅ {len(raw_docs)} documents loaded')
print(f'\nSample:\n{raw_docs[0].page_content[:200]}...')

---
## 🔪 Cell 3 — Chunk the Documents

We split each document into smaller pieces called **chunks** — like cutting a book into paragraphs before indexing it.

Two settings to control:
- `chunk_size` — how long each piece can be (in characters)
- `chunk_overlap` — how many characters the next chunk repeats from the end of the previous one

Overlap exists so that important sentences don't get cut in half between two chunks.

🎯 **You will experiment with these in Cell 7.**

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Change these in Cell 7 ────────────────────────────────────────────────
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 50
# ─────────────────────────────────────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(raw_docs)

print(f'✅ {len(raw_docs)} documents → {len(chunks)} chunks')
print(f'Average chunk: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters')
print(f'\n--- Chunk 0 ---\n{chunks[0].page_content}')
print(f'\n--- Chunk 1 (notice the repeated overlap at the start) ---\n{chunks[1].page_content[:200]}')

---
## 🗄 Cell 4 — Embed Chunks and Store Them

Each chunk gets converted into a list of numbers called a **vector** or **embedding**.
Two chunks with similar meaning will have vectors that are close together in space.
That is what makes semantic search possible.

We store all vectors in **ChromaDB** — a lightweight in-memory database. Free, no key needed.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print('⏳ Loading embedding model (downloads ~80MB on first run)...')
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name='vois_handbook',
)

print(f'✅ {vectorstore._collection.count()} chunks embedded and stored')

---
## 🔍 Cell 5 — Ask a Question (Retrieval Only)

Before calling any LLM, see which chunks the system finds.
This is the most important step — if the retrieval is wrong, the answer will be wrong too.

In [ ]:
# ── Try different questions ───────────────────────────────────────────────
QUESTION = 'How many days of annual leave do I get?'
# QUESTION = 'Can I work from home?'
# QUESTION = 'What happens if I lose my laptop?'
# QUESTION = 'How do I get promoted?'
# QUESTION = 'What is the training budget?'
TOP_K = 3
# ─────────────────────────────────────────────────────────────────────────

retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': TOP_K})
retrieved_chunks = retriever.invoke(QUESTION)

print(f'Question: "{QUESTION}"')
print(f'\nTop {TOP_K} chunks found:\n')
for i, chunk in enumerate(retrieved_chunks):
    print(f'--- Result {i+1}  [{chunk.metadata["section"]}] ---')
    print(chunk.page_content[:300])
    print()

---
## 🤖 Cell 6 — Generate an Answer with GPT-4o

We take the retrieved chunks, build a prompt, and ask GPT-4o to answer.
The system prompt tells the model to **only use what is in the chunks** — no making things up.

> No API key? The cell will show you the full prompt that would be sent, so you can still follow along.

In [ ]:
# ── Paste your OpenAI key here (or leave to skip) ─────────────────────────
OPENAI_API_KEY = 'sk-...'  # replace with your key
# ─────────────────────────────────────────────────────────────────────────

context = '\n\n'.join(f'[{c.metadata["section"]}]\n{c.page_content}' for c in retrieved_chunks)

system_prompt = (
    'You are a helpful VOIS HR assistant. '
    'Answer ONLY from the provided context. '
    'If the answer is not there, say: I do not have that information. '
    'Always say which section your answer comes from.'
)
user_prompt = f'Context:\n{context}\n\nQuestion: {QUESTION}'

if OPENAI_API_KEY.startswith('sk-') and len(OPENAI_API_KEY) > 10:
    from openai import OpenAI
    client   = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'system', 'content': system_prompt},
                  {'role': 'user',   'content': user_prompt}],
        temperature=0, max_tokens=300,
    )
    print(f'Question: {QUESTION}\n')
    print(f'Answer:\n{response.choices[0].message.content}')
else:
    print('No API key — showing the prompt that would be sent to GPT-4o:\n')
    print('SYSTEM:', system_prompt)
    print('\nUSER (first 600 chars):')
    print(user_prompt[:600], '...')

---
## 🎯 Cell 7 — TRY THIS: Does Chunk Size Matter?

**Go back to Cell 3**, change `CHUNK_SIZE` to `256` or `1024`, then re-run cells 3 → 4 → 5.

Then run this cell to see which chunk size finds the right answer most often:

In [ ]:
test_questions = [
    ('How many days of annual leave do I get?',         '21 working days'),
    ('Can I work from home every day?',                  'up to 3 days per week'),
    ('What is my training budget?',                      'EGP 15,000'),
    ('What happens if I get a bad performance review?',  'Performance Improvement Plan'),
    ('Is dental covered by my insurance?',               'dental'),
]

hits = 0
print(f'Chunk size: {CHUNK_SIZE}  |  Overlap: {CHUNK_OVERLAP}\n')
print(f'{"Question":<52}  {"Found?"}')
print('-' * 65)
for q, keyword in test_questions:
    docs  = retriever.invoke(q)
    found = keyword.lower() in ' '.join(d.page_content for d in docs).lower()
    hits += found
    print(f'{q[:50]:<52}  {"✅" if found else "❌"}')
print(f'\nResult: {hits} / {len(test_questions)} questions answered correctly')
print('\nNow try chunk_size=256 or 1024 — which score changes?')

---
## 🔀 Cell 8 — Hybrid Search

### Why vector search alone sometimes fails

Vector search is great at understanding **meaning**. But it can miss **exact words**.

**Example:** Search for `"EGP 15,000 training budget"`.
Vector search might return a chunk about *general learning programs* because the meaning feels similar — even though it does not contain the actual number.

### The idea: use two search methods at the same time

Think of it like searching for a book in a library two ways at once:
- 🔵 **Vector search** — asks "what is this *about*?" — good for meaning and paraphrases
- 🟡 **Keyword search (BM25)** — asks "does this contain these *exact words*?" — good for names, codes, numbers

We combine both scores into one final score:
```
final_score = 0.7 × vector_score  +  0.3 × keyword_score
```
The `0.7 / 0.3` split is a starting point you can tune.

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# Build keyword index
bm25 = BM25Okapi([c.page_content.lower().split() for c in chunks])

def hybrid_search(query, k=3, vector_weight=0.7):
    keyword_weight = 1 - vector_weight

    # Vector scores
    v_results = vectorstore.similarity_search_with_score(query, k=len(chunks))
    v_scores  = {doc.page_content: 1 - score for doc, score in v_results}

    # Keyword scores (normalised to 0-1)
    kw = bm25.get_scores(query.lower().split())
    kw = kw / (kw.max() + 1e-9)

    # Combine
    combined = []
    for i, chunk in enumerate(chunks):
        score = vector_weight * v_scores.get(chunk.page_content, 0) + keyword_weight * kw[i]
        combined.append((chunk, score))

    combined.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in combined[:k]]


# Test with a query that contains an exact number
test_query = 'EGP 15,000 certification reimbursement'

vector_result = retriever.invoke(test_query)
hybrid_result = hybrid_search(test_query, k=3)

print(f'Query: "{test_query}"\n')
print('🔵 Vector only — top result section:', vector_result[0].metadata['section'])
print(vector_result[0].page_content[:200])

print('\n🟢 Hybrid — top result section:', hybrid_result[0].metadata['section'])
print(hybrid_result[0].page_content[:200])

print('\n💡 If the hybrid result contains "EGP 15,000", it found the right chunk. Vector-only might not.')

---
## 🏆 Cell 9 — Reranking

### The problem that retrieval still has

Both vector search and hybrid search score each chunk by itself — they do not look at the question and the chunk *together*.

Imagine a very thorough librarian. When you ask a question, they do not just scan the shelf labels. They pick up a book, read a paragraph, and decide: *"does this actually answer what the person asked?"*

That is what a **reranker** does.

### How it works (step by step, no jargon)

```
Step 1 — Retrieve a bigger pool cheaply (top 10)
           ↓
Step 2 — For each chunk, the reranker reads:
           [Question] + [Chunk text]
           and gives it a relevance score
           ↓
Step 3 — Re-sort by that score, keep the best 3
           ↓
Step 4 — Send those 3 to the LLM
```

It is slower than vector search, so we only run it on a small shortlist — not all chunks.

In [ ]:
from sentence_transformers import CrossEncoder

print('⏳ Loading reranker (~100MB one-time download)...')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('✅ Reranker ready')

def rerank(query, candidates, top_k=3):
    pairs  = [[query, doc.page_content] for doc in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]], [s for _, s in ranked[:top_k]]


# Get a bigger pool (10 results), then let the reranker pick the best 3
broad       = vectorstore.as_retriever(search_kwargs={'k': 10}).invoke(QUESTION)
ranked_docs, ranked_scores = rerank(QUESTION, broad, top_k=3)

print(f'\nQuestion: "{QUESTION}"')
print(f'Retrieved {len(broad)} candidates → reranker selected the best 3:\n')
for i, (doc, score) in enumerate(zip(ranked_docs, ranked_scores)):
    print(f'Rank {i+1}  (relevance: {score:.2f})  [{doc.metadata["section"]}]')
    print(doc.page_content[:200])
    print()
print('Higher score = the reranker is more confident this chunk answers the question.')

---
## 🎯 Cell 10 — TRY THIS: Vector vs Hybrid vs Reranked Side by Side

In [ ]:
print(f'{"Question":<52}  {"Vector":>7}  {"Hybrid":>7}  {"Reranked":>9}')
print('-' * 82)

v_hits = h_hits = r_hits = 0
for q, keyword in test_questions:
    v_docs  = retriever.invoke(q)
    h_docs  = hybrid_search(q, k=3)
    pool    = vectorstore.as_retriever(search_kwargs={'k': 10}).invoke(q)
    r_docs, _ = rerank(q, pool, top_k=3)

    v = keyword.lower() in ' '.join(d.page_content for d in v_docs).lower()
    h = keyword.lower() in ' '.join(d.page_content for d in h_docs).lower()
    r = keyword.lower() in ' '.join(d.page_content for d in r_docs).lower()
    v_hits += v; h_hits += h; r_hits += r

    print(f'{q[:50]:<52}  {"✅" if v else "❌":>7}  {"✅" if h else "❌":>7}  {"✅" if r else "❌":>9}')

n = len(test_questions)
print('-' * 82)
print(f'{"Correct":52}  {v_hits}/{n:>5}     {h_hits}/{n:>5}     {r_hits}/{n:>7}')
print('\nWhich method finds the most correct answers on this knowledge base?')

---
## 📊 Cell 11 — Build Evaluation Dataset

RAGAS needs four things per question:
- The question
- The chunks that were retrieved
- The answer the system gave
- The correct answer (so it can check)

In [ ]:
from datasets import Dataset

eval_pairs = [
    {'question': 'How many days of annual leave do VOIS employees get?',      'ground_truth': 'VOIS employees are entitled to 21 working days of annual leave per calendar year.'},
    {'question': 'What is the maximum number of remote work days per week?',   'ground_truth': 'Employees may work remotely up to 3 days per week.'},
    {'question': 'What is the annual training budget for VOIS employees?',     'ground_truth': 'VOIS employees are entitled to a personal training budget of EGP 15,000 per year.'},
    {'question': 'When do performance reviews take place at VOIS?',            'ground_truth': 'Reviews take place in June (mid-year) and December (year-end).'},
    {'question': 'What happens if an employee loses their company laptop?',    'ground_truth': 'Loss due to negligence may result in a financial deduction from the final settlement.'},
]

eval_data = {'question': [], 'answer': [], 'contexts': [], 'ground_truth': []}

for pair in eval_pairs:
    q    = pair['question']
    docs = retriever.invoke(q)
    ctx  = [d.page_content for d in docs]

    if OPENAI_API_KEY.startswith('sk-') and len(OPENAI_API_KEY) > 10:
        from openai import OpenAI
        resp = OpenAI(api_key=OPENAI_API_KEY).chat.completions.create(
            model='gpt-4o',
            messages=[{'role': 'system', 'content': 'Answer based only on the context. Be concise.'},
                      {'role': 'user',   'content': f'Context:\n{chr(10).join(ctx)}\n\nQuestion: {q}'}],
            temperature=0, max_tokens=150,
        )
        ans = resp.choices[0].message.content
    else:
        ans = ctx[0][:250] if ctx else 'No context found.'

    eval_data['question'].append(q)
    eval_data['answer'].append(ans)
    eval_data['contexts'].append(ctx)
    eval_data['ground_truth'].append(pair['ground_truth'])

eval_dataset = Dataset.from_dict(eval_data)
print(f'✅ {len(eval_dataset)} evaluation rows ready')

---
## 📊 Cell 12 — Run RAGAS Evaluation

Three scores, all between 0 and 1:

| Score | What it checks | Target |
|-------|---------------|--------|
| **faithfulness** | Did the answer only use what was in the retrieved chunks? | ≥ 0.85 |
| **answer_relevancy** | Did the answer actually address the question? | ≥ 0.80 |
| **context_precision** | Were the most relevant chunks at the top of the results? | ≥ 0.75 |

> RAGAS uses an LLM to judge, so it needs the OpenAI key.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

print('⏳ Running RAGAS (1-2 min)...')
try:
    results = evaluate(dataset=eval_dataset,
                       metrics=[faithfulness, answer_relevancy, context_precision])

    print('\n' + '=' * 50)
    print('         RAGAS RESULTS')
    print('=' * 50)
    print(f'  faithfulness       : {results["faithfulness"]:.3f}   (target ≥ 0.85)')
    print(f'  answer_relevancy   : {results["answer_relevancy"]:.3f}   (target ≥ 0.80)')
    print(f'  context_precision  : {results["context_precision"]:.3f}   (target ≥ 0.75)')
    print('=' * 50)
    print('\nWhat to try if a score is low:')
    if results['context_precision'] < 0.75:
        print('  context_precision low → go back to Cell 3, try a smaller chunk_size')
    if results['faithfulness'] < 0.85:
        print('  faithfulness low → enable reranking (Cell 9) to pass better chunks to the LLM')
    if results['answer_relevancy'] < 0.80:
        print('  answer_relevancy low → tighten the system prompt in Cell 6')
    if all(results[m] >= t for m, t in [('faithfulness', 0.85), ('answer_relevancy', 0.80), ('context_precision', 0.75)]):
        print('  ✅ All scores within target — nice work!')

except Exception as e:
    print('ℹ️  RAGAS needs a valid OpenAI key to run.')
    print(f'   ({e})')
    print('\n   Without a key, you would see something like:')
    print('     faithfulness      : 0.84')
    print('     answer_relevancy  : 0.81')
    print('     context_precision : 0.72  ← suggests trying a smaller chunk_size')

---
## 🏗 Cell 13 — Moving to Azure (Production)

Everything we built runs locally in Colab. Moving to Azure requires changing only two lines.

```python
# BEFORE (Colab / local)
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(chunks, embedding_model)

# AFTER (Azure AI Search)
from langchain_community.vectorstores import AzureSearch
vectorstore = AzureSearch(
    azure_search_endpoint='https://your-service.search.windows.net',
    azure_search_key='your-key',
    index_name='vois-rag',
    embedding_function=embedding_model.embed_query,
    search_type='hybrid',   # ← hybrid search is one parameter in Azure
)

# BEFORE (HuggingFace free model)
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# AFTER (Azure OpenAI)
from langchain_openai import AzureOpenAIEmbeddings
embedding_model = AzureOpenAIEmbeddings(
    azure_deployment='text-embedding-3-large',
    azure_endpoint='https://your-openai.openai.azure.com',
    api_key='your-azure-openai-key',
)
```

Chunking, retrieval, generation, and RAGAS evaluation all stay exactly the same.

---
## 📚 References
- RAGAS: https://arxiv.org/abs/2309.15217
- RAG Survey (Gao et al., 2023): https://arxiv.org/abs/2312.10997
- LangChain docs: https://python.langchain.com
- sentence-transformers: https://sbert.net

---
*VOIS AI Team · April 2026 · Sara Zayan & Youssef Mazen*